In [16]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, average_precision_score

def resolve_device(prefer: Optional[str] = None) -> torch.device:
    # prefer: None or 'cpu'/'mps'/'cuda'
    if prefer is not None:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = resolve_device()
print("Torch:", torch.__version__)
print("Device:", DEVICE)

Torch: 2.9.0
Device: mps


In [17]:
def read_fasta(path: Path) -> Tuple[List[str], List[str]]:
    headers, seqs = [], []
    cur = []
    head = None
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if head is not None:
                    seqs.append("".join(cur).upper())
                    cur = []
                head = line[1:].strip()
                headers.append(head)
            else:
                cur.append(line)
        if head is not None:
            seqs.append("".join(cur).upper())
    assert len(headers) == len(seqs)
    return headers, seqs

def load_binary_feature_labels(path: Path) -> Dict[str, int]:
    label_dict = {}
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 20:
                continue
            header = parts[0][1:] if parts[0].startswith(">") else parts[0]
            label_dict[header] = 1 if parts[1].upper() == "TRUE" else 0
    return label_dict

def header_to_tag(header: str) -> str:
    left = header.split("#")[0]
    return left.split("::")[0]

def build_multiclass_labels_from_headers(headers: List[str]):
    tags = [header_to_tag(h) for h in headers]
    uniq = sorted(set(tags))
    tag_to_id = {t:i for i,t in enumerate(uniq)}
    y = np.array([tag_to_id[t] for t in tags], dtype=np.int64)
    return y, tag_to_id, uniq

def make_binary_labels_from_tags(headers: List[str], negative_tag: str = "None") -> np.ndarray:
    tags = [header_to_tag(h) for h in headers]
    return np.array([0 if t == negative_tag else 1 for t in tags], dtype=np.int64)


In [18]:
# ASCII -> {0,1,2,3,4} for A,C,G,T,other
_ASCII_MAP = np.full(256, 4, dtype=np.uint8)
for ch,val in [("A",0),("C",1),("G",2),("T",3),("a",0),("c",1),("g",2),("t",3)]:
    _ASCII_MAP[ord(ch)] = val

_COMP = np.array([3,2,1,0], dtype=np.uint8)  # A<->T, C<->G in 0..3

def kmer_code_forward(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4:
        code = (code << 2) | int(v)
    return code

def kmer_code_rc(arr4: np.ndarray) -> int:
    code = 0
    for v in arr4[::-1]:
        code = (code << 2) | int(_COMP[v])
    return code

def canonical_kmer_code(arr4: np.ndarray) -> int:
    c1 = kmer_code_forward(arr4)
    c2 = kmer_code_rc(arr4)
    return c1 if c1 < c2 else c2

def hash_u32(x: int, dim: int) -> int:
    z = (x * 0x9E3779B97F4A7C15) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 33)
    z = (z * 0xC2B2AE3D27D4EB4F) & 0xFFFFFFFFFFFFFFFF
    z ^= (z >> 29)
    return int(z % dim)

@dataclass
class KmerWindowFeaturizer:
    k: int = 7
    dim: int = 2048
    window: int = 512
    stride: int = 256
    add_pos: bool = True
    l2_normalize: bool = True

    def featurize_sequence(self, seq: str):
        arr = _ASCII_MAP[np.frombuffer(seq.encode("ascii", "ignore"), dtype=np.uint8)]
        L = int(arr.size)
        if L == 0:
            X = np.zeros((1, self.dim + (1 if self.add_pos else 0)), dtype=np.float32)
            return X, np.array([0], dtype=np.int64)

        if L <= self.window:
            starts = np.array([0], dtype=np.int64)
        else:
            starts = np.arange(0, L - self.window + 1, self.stride, dtype=np.int64)
            if starts.size == 0:
                starts = np.array([0], dtype=np.int64)

        out_dim = self.dim + (1 if self.add_pos else 0)
        X = np.zeros((starts.size, out_dim), dtype=np.float32)

        for wi, st in enumerate(starts):
            en = min(st + self.window, L)
            sub = arr[st:en]
            counts = np.zeros(self.dim, dtype=np.float32)
            total = 0

            k = self.k
            if sub.size >= k:
                for i in range(0, sub.size - k + 1):
                    kmer = sub[i:i+k]
                    if np.any(kmer == 4):
                        continue
                    code = canonical_kmer_code(kmer)
                    j = hash_u32(code, self.dim)
                    counts[j] += 1.0
                    total += 1

            if total > 0:
                counts /= float(total)

            if self.l2_normalize:
                nrm = np.linalg.norm(counts)
                if nrm > 0:
                    counts /= nrm

            if self.add_pos:
                center = (st + en) / 2.0
                pos = center / max(1.0, float(L))
                X[wi, :-1] = counts
                X[wi, -1] = pos
            else:
                X[wi, :] = counts

        return X, starts


In [19]:
@dataclass
class GraphSample:
    x: torch.Tensor
    edge_index: torch.Tensor
    y: torch.Tensor
    header: str

def build_chain_edge_index(n: int, undirected: bool = True, self_loops: bool = True) -> torch.Tensor:
    edges = []
    if n > 1:
        src = np.arange(n-1, dtype=np.int64)
        dst = np.arange(1, n, dtype=np.int64)
        edges.append((src, dst))
        if undirected:
            edges.append((dst, src))
    if self_loops:
        idx = np.arange(n, dtype=np.int64)
        edges.append((idx, idx))
    if not edges:
        ei = np.zeros((2,0), dtype=np.int64)
    else:
        s = np.concatenate([e[0] for e in edges])
        d = np.concatenate([e[1] for e in edges])
        ei = np.stack([s,d], axis=0)
    return torch.from_numpy(ei)

class WindowGraphDataset(torch.utils.data.Dataset):
    def __init__(self, headers: List[str], sequences: List[str], y: np.ndarray, featurizer: KmerWindowFeaturizer):
        self.headers=headers
        self.seqs=sequences
        self.y=y.astype(np.int64)
        self.feat=featurizer

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx: int) -> GraphSample:
        h=self.headers[idx]
        seq=self.seqs[idx]
        X,_=self.feat.featurize_sequence(seq)
        x=torch.from_numpy(X).to(torch.float32)
        ei=build_chain_edge_index(x.size(0), undirected=True, self_loops=True)
        y=torch.tensor(int(self.y[idx]), dtype=torch.int64)
        return GraphSample(x=x, edge_index=ei, y=y, header=h)

def collate_graphs(batch: List[GraphSample]):
    xs=[]; eis=[]; ys=[]; headers=[]
    batch_vecs=[]
    node_offset=0
    for gi,s in enumerate(batch):
        n=s.x.size(0)
        xs.append(s.x)
        eis.append(s.edge_index + node_offset)
        ys.append(s.y)
        headers.append(s.header)
        batch_vecs.append(torch.full((n,), gi, dtype=torch.int64))
        node_offset += n
    x=torch.cat(xs, dim=0)
    edge_index=torch.cat(eis, dim=1) if eis else torch.zeros((2,0),dtype=torch.int64)
    batch_vec=torch.cat(batch_vecs, dim=0)
    y=torch.stack(ys, dim=0)
    return x, edge_index, batch_vec, y, headers

def scatter_mean(x: torch.Tensor, idx: torch.Tensor, dim_size: int) -> torch.Tensor:
    out=torch.zeros((dim_size, x.size(1)), device=x.device, dtype=x.dtype)
    out.index_add_(0, idx, x)
    cnt=torch.bincount(idx, minlength=dim_size).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
    return out/cnt


In [20]:
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index[0], edge_index[1]
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        deg = torch.bincount(dst, minlength=x.size(0)).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
        agg = agg / deg
        h = self.lin_self(x) + self.lin_neigh(agg)
        h = F.relu(h)
        return self.dropout(h)

class GNNClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden: int, num_classes: int, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        layers=[]
        d=in_dim
        for _ in range(n_layers):
            layers.append(GraphSAGELayer(d, hidden, dropout=dropout))
            d=hidden
        self.layers=nn.ModuleList(layers)
        self.head=nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch_vec: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, edge_index)
        B = int(batch_vec.max().item()) + 1 if batch_vec.numel() else 0
        g = scatter_mean(x, batch_vec, dim_size=B)
        return self.head(g)


In [21]:
def compute_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / counts
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def predict_loader(model: nn.Module, loader, device: torch.device):
    model.eval()
    logits_list=[]; y_list=[]; headers=[]
    for x, edge_index, batch_vec, y, h in loader:
        x=x.to(device); edge_index=edge_index.to(device); batch_vec=batch_vec.to(device)
        logits=model(x, edge_index, batch_vec)
        logits_list.append(logits.detach().cpu())
        y_list.append(y.detach().cpu())
        headers.extend(h)
    logits=torch.cat(logits_list, dim=0) if logits_list else torch.empty((0,1))
    y=torch.cat(y_list, dim=0) if y_list else torch.empty((0,), dtype=torch.int64)
    return logits, y, headers

def evaluate_multiclass(logits: torch.Tensor, y: torch.Tensor) -> Dict[str, float]:
    y_np=y.numpy()
    probs=torch.softmax(logits, dim=1).numpy()
    pred=probs.argmax(axis=1)
    out={}
    out['acc']=accuracy_score(y_np, pred) if len(y_np) else float('nan')
    out['balanced_acc']=balanced_accuracy_score(y_np, pred) if len(y_np) else float('nan')
    try:
        out['auroc_macro_ovr']=float(roc_auc_score(y_np, probs, multi_class='ovr', average='macro'))
    except Exception:
        out['auroc_macro_ovr']=float('nan')
    try:
        Y=np.zeros_like(probs)
        Y[np.arange(len(y_np)), y_np]=1.0
        out['auprc_macro']=float(average_precision_score(Y, probs, average='macro'))
    except Exception:
        out['auprc_macro']=float('nan')
    return out

def run_train_gnn(
    fasta_path: Path,
    label_mode: str = "multiclass",
    label_path: Optional[Path] = None,
    negative_tag: str = "None",
    k: int = 7,
    dim: int = 2048,
    window: int = 512,
    stride: int = 256,
    hidden: int = 128,
    n_layers: int = 3,
    dropout: float = 0.1,
    batch_size: int = 16,
    epochs: int = 20,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 5,
    subset_size: Optional[int] = 20000,
    random_state: int = 42,
    device: Optional[torch.device] = None,
    num_workers: int = 0,
):
    device = resolve_device(None if device is None else str(device))
    print("Using device:", device)

    headers, sequences = read_fasta(Path(fasta_path))
    print("Loaded sequences:", len(sequences))

    if subset_size is not None and subset_size < len(sequences):
        rng=np.random.default_rng(random_state)
        idx=rng.permutation(len(sequences))[:subset_size]
        idx.sort()
        headers=[headers[i] for i in idx]
        sequences=[sequences[i] for i in idx]
        print("Subset size:", len(sequences))

    id_to_tag=None
    if label_mode == "multiclass":
        y, tag_to_id, id_to_tag = build_multiclass_labels_from_headers(headers)
        num_classes=len(tag_to_id)
        print("Num classes:", num_classes)
    elif label_mode == "binary_none_tag":
        y = make_binary_labels_from_tags(headers, negative_tag=negative_tag)
        num_classes=2
    elif label_mode == "binary_feature":
        assert label_path is not None
        d = load_binary_feature_labels(Path(label_path))
        keep_h=[]; keep_s=[]; y_list=[]
        for h,s in zip(headers,sequences):
            if h in d:
                keep_h.append(h); keep_s.append(s); y_list.append(d[h])
        headers, sequences = keep_h, keep_s
        y=np.array(y_list, dtype=np.int64)
        num_classes=2
    else:
        raise ValueError("Unknown label_mode")

    idx_all=np.arange(len(sequences))
    idx_tr, idx_te = train_test_split(idx_all, test_size=0.2, random_state=random_state, stratify=y)
    idx_tr, idx_va = train_test_split(idx_tr, test_size=0.2, random_state=random_state, stratify=y[idx_tr])

    feat = KmerWindowFeaturizer(k=k, dim=dim, window=window, stride=stride, add_pos=True, l2_normalize=True)
    ds_tr = WindowGraphDataset([headers[i] for i in idx_tr], [sequences[i] for i in idx_tr], y[idx_tr], feat)
    ds_va = WindowGraphDataset([headers[i] for i in idx_va], [sequences[i] for i in idx_va], y[idx_va], feat)
    ds_te = WindowGraphDataset([headers[i] for i in idx_te], [sequences[i] for i in idx_te], y[idx_te], feat)

    loader_tr = torch.utils.data.DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_graphs)
    loader_va = torch.utils.data.DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_graphs)
    loader_te = torch.utils.data.DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_graphs)

    in_dim = dim + 1
    model = GNNClassifier(in_dim=in_dim, hidden=hidden, num_classes=num_classes, n_layers=n_layers, dropout=dropout).to(device)
    class_w = compute_class_weights(y[idx_tr], num_classes).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state=None
    best_score=-float("inf")
    bad=0
    history={"train_loss":[], "val_loss":[], "val_acc":[], "val_bacc":[], "val_auroc":[], "val_auprc":[]}

    for ep in range(1, epochs+1):
        model.train()
        running=0.0; n_graphs=0
        for x, edge_index, batch_vec, yb, _ in loader_tr:
            x=x.to(device); edge_index=edge_index.to(device); batch_vec=batch_vec.to(device); yb=yb.to(device)
            logits=model(x, edge_index, batch_vec)
            loss=loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            running += float(loss.item()) * yb.size(0)
            n_graphs += yb.size(0)
        train_loss = running / max(1, n_graphs)

        model.eval()
        val_running=0.0; val_n=0
        all_logits=[]; all_y=[]
        with torch.no_grad():
            for x, edge_index, batch_vec, yb, _ in loader_va:
                x=x.to(device); edge_index=edge_index.to(device); batch_vec=batch_vec.to(device); yb=yb.to(device)
                logits=model(x, edge_index, batch_vec)
                loss=loss_fn(logits, yb)
                val_running += float(loss.item()) * yb.size(0)
                val_n += yb.size(0)
                all_logits.append(logits.detach().cpu())
                all_y.append(yb.detach().cpu())
        val_loss = val_running / max(1, val_n)
        logits_va = torch.cat(all_logits, dim=0) if all_logits else torch.empty((0,num_classes))
        y_va = torch.cat(all_y, dim=0) if all_y else torch.empty((0,),dtype=torch.int64)
        m = evaluate_multiclass(logits_va, y_va)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(m["acc"])
        history["val_bacc"].append(m["balanced_acc"])
        history["val_auroc"].append(m["auroc_macro_ovr"])
        history["val_auprc"].append(m["auprc_macro"])

        print(f"Epoch {ep:02d} | train={train_loss:.4f} | val={val_loss:.4f} | acc={m['acc']:.4f} | bacc={m['balanced_acc']:.4f} | auroc={m['auroc_macro_ovr']:.4f} | auprc={m['auprc_macro']:.4f}")

        score = m["auprc_macro"]
        if np.isnan(score):
            score = m["balanced_acc"]

        if score > best_score + 1e-4:
            best_score=score
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)

    logits_te, y_te, headers_te = predict_loader(model, loader_te, device)
    test_metrics = evaluate_multiclass(logits_te, y_te)
    print("Test metrics:", test_metrics)

    return {"model":model, "featurizer":feat, "history":history, "test":{"logits":logits_te, "y":y_te, "headers":headers_te, "metrics":test_metrics}, "id_to_tag":id_to_tag, "device":str(device)}


In [22]:
@torch.no_grad()
def predict_sequence(model: nn.Module, featurizer: KmerWindowFeaturizer, seq: str, device: torch.device):
    X,_ = featurizer.featurize_sequence(seq)
    x = torch.from_numpy(X).to(device)
    edge_index = build_chain_edge_index(x.size(0), undirected=True, self_loops=True).to(device)
    batch_vec = torch.zeros((x.size(0),), dtype=torch.int64, device=device)
    logits = model(x, edge_index, batch_vec).squeeze(0)
    return torch.softmax(logits, dim=0).detach().cpu().numpy()

def topk_classes(probs: np.ndarray, id_to_tag=None, k: int = 5):
    idx = np.argsort(-probs)[:k]
    return [((id_to_tag[i] if id_to_tag else int(i)), float(probs[i])) for i in idx]

def find_misclassified(logits: torch.Tensor, y: torch.Tensor, headers: List[str], id_to_tag=None, max_rows: int = 50):
    probs = torch.softmax(logits, dim=1).numpy()
    pred = probs.argmax(axis=1)
    y_np = y.numpy()
    bad = np.where(pred != y_np)[0]
    rows=[]
    for i in bad[:max_rows]:
        rows.append({"header":headers[i], "true": (id_to_tag[y_np[i]] if id_to_tag else int(y_np[i])), "pred": (id_to_tag[pred[i]] if id_to_tag else int(pred[i])), "p_pred": float(probs[i, pred[i]])})
    return rows, int(bad.size)

def save_checkpoint(path: Path, model: nn.Module, featurizer: KmerWindowFeaturizer, id_to_tag=None):
    ckpt={"model_state": model.state_dict(), "featurizer": featurizer.__dict__, "id_to_tag": id_to_tag}
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(ckpt, path)
    print("Saved:", path)

def load_checkpoint(path: Path, hidden: int, n_layers: int, dropout: float = 0.1, device: Optional[torch.device] = None):
    device = resolve_device(None if device is None else str(device))
    ckpt=torch.load(path, map_location="cpu")
    feat=KmerWindowFeaturizer(**ckpt["featurizer"])
    id_to_tag=ckpt.get("id_to_tag", None)
    state=ckpt["model_state"]
    num_classes = state["head.3.weight"].shape[0]
    in_dim = feat.dim + (1 if feat.add_pos else 0)
    model = GNNClassifier(in_dim=in_dim, hidden=hidden, num_classes=num_classes, n_layers=n_layers, dropout=dropout)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model, feat, id_to_tag, device


In [23]:
FASTA_PATH = Path("../data/vgp/all_vgp_tes.fa")

results = run_train_gnn(
    fasta_path=FASTA_PATH,
    label_mode="multiclass",
    subset_size=20000,
    batch_size=16,
    epochs=25,
    patience=5,
    dim=2048,
    window=512,
    stride=256,
)

Using device: mps
Loaded sequences: 135751
Subset size: 20000
Num classes: 20000


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.